In [2]:
import pandas as pd
df_raw= pd.read_parquet("output_manual/bioclip_full_run/bioclip2_all_candidate.parquet")
df_raw.head()

,model_kingdom_name,model_phylum_name,model_class_name,model_order_name,model_family_name,model_genus_name,model_species_epithet,model_species_name,model_common_name,model_score,...,box_area_frac,aspect_ratio,edge_proximity_frac,center_offset_frac,score_rank_gap,crop_x0,crop_y0,crop_x1,crop_y1,was_out_of_bounds
0,Animalia,Chordata,Aves,Cuculiformes,Cuculidae,Geococcyx,californianus,Geococcyx californianus,Greater Roadrunner,0.968591,...,0.163672,2.4779,0.080413,0.1253,NaN,346,387,920,619,False
1,Animalia,Chordata,Aves,Cuculiformes,Cuculidae,Geococcyx,velox,Geococcyx velox,Lesser roadrunner,0.021866,...,0.163672,2.4779,0.080413,0.1253,NaN,346,387,920,619,False
2,Animalia,Chordata,Aves,Passeriformes,Troglodytidae,Campylorhynchus,brunneicapillus,Campylorhynchus brunneicapillus,Cactus Wren,0.000245,...,0.163672,2.4779,0.080413,0.1253,NaN,346,387,920,619,False
3,Animalia,Arthropoda,Insecta,Orthoptera,Acrididae,Schistocerca,gregaria,Schistocerca gregaria,Desert locust,0.000141,...,0.163672,2.4779,0.080413,0.1253,NaN,346,387,920,619,False
4,Animalia,Chordata,Squamata,,Teiidae,Aspidoscelis,tigris,Aspidoscelis tigris,Tiger whiptail,0.000115,...,0.163672,2.4779,0.080413,0.1253,NaN,346,387,920,619,False


In [3]:
df_raw.columns

Index(['model_kingdom_name', 'model_phylum_name', 'model_class_name',
       'model_order_name', 'model_family_name', 'model_genus_name',
       'model_species_epithet', 'model_species_name', 'model_common_name',
       'model_score', 'model_prediction_rank', 'obs_id', 'photo_id',
       'photo_license_code', 'photo_url', 'obs_url', 'quality_grade',
       'obs_license_code', 'predator_prey_role', 'scientific_name',
       'taxon_rank', 'common_name', 'latitude', 'longitude',
       'positional_accuracy', 'photo_count', 'kingdom_name', 'phylum_name',
       'class_name', 'order_name', 'family_name', 'genus_name', 'species_name',
       'subspecies_name', 'subfamily_name', 'hybrid_name', 'complex_name',
       'form_name', 'subgenus_name', 'variety_name', 'tribe_name', 'country',
       'state', 'city', 'file_name', 'image_byte_size', 'image_local_path',
       'image_download_url', 'grounding_dino_prompt', 'source_row',
       'img_width', 'img_height', 'n_detections', 'max_score', 'me

## Rank level match

In [4]:
ranks = [
    'kingdom',
    'phylum',
    'class',
    "order",
    "family",
    "genus",
    "species"
]

In [6]:

def rank_match(rank):
    match_col = f"{rank}_match"
    source_col = f"{rank}_name"
    model_col = f"model_{rank}_name"
    df_raw[match_col] = df_raw[source_col].astype(str) == df_raw[model_col].astype(str)
    print(f"Added {match_col} to the df_raw")
    if match_col in df_raw.columns: 
        print(f"{match_col} column exists ✔️")
    else:
        print(f"{match_col} column does not exist ❌")
    print("-"*20)

for rank in ranks:
    rank_match(rank)

Added kingdom_match to the df_raw
Column does not exist ❌
--------------------
Added phylum_match to the df_raw
Column does not exist ❌
--------------------
Added class_match to the df_raw
Column does not exist ❌
--------------------
Added order_match to the df_raw
Column does not exist ❌
--------------------
Added family_match to the df_raw
Column does not exist ❌
--------------------
Added genus_match to the df_raw
Column does not exist ❌
--------------------
Added species_match to the df_raw
Column does not exist ❌
--------------------


In [9]:
for rank in ranks:
    match_col = f"{rank}_match"
    if match_col in df_raw.columns: 
        print("Column exists ✔️")
    else:
        print("Column does not exist ❌")
    print("-"*20)

Column exists ✔️
--------------------
Column exists ✔️
--------------------
Column exists ✔️
--------------------
Column exists ✔️
--------------------
Column exists ✔️
--------------------
Column exists ✔️
--------------------
Column exists ✔️
--------------------


In [15]:
# Image-level BioCLIP pipeline evaluation

def rank_level_accuracy(rank):
    match_col = f"{rank}_match"
    image_results = (
        df_raw.groupby("file_name")[match_col]
        .any()
        .reset_index(name="identified")
    )
    
    # Counts
    n_total = len(image_results)
    n_identified = image_results["identified"].sum()
    n_failed = n_total - n_identified
    
    print(f"{rank} level match")
    print(f"Total images:      {n_total}")
    print(f"Identified:        {n_identified}")
    print(f"Failed:            {n_failed}")
    print(f"Accuracy:          {n_identified / n_total:.4f}")
    print(f"Accuracy (%):      {100 * n_identified / n_total:.2f}%")
    print("----------------------------------------------------------------------------")


In [16]:
for rank in ranks:
    rank_level_accuracy(rank)

kingdom level match
Total images:      22524
Identified:        22435
Failed:            89
Accuracy:          0.9960
Accuracy (%):      99.60%
----------------------------------------------------------------------------
phylum level match
Total images:      22524
Identified:        22092
Failed:            432
Accuracy:          0.9808
Accuracy (%):      98.08%
----------------------------------------------------------------------------
class level match
Total images:      22524
Identified:        19169
Failed:            3355
Accuracy:          0.8510
Accuracy (%):      85.10%
----------------------------------------------------------------------------
order level match
Total images:      22524
Identified:        18522
Failed:            4002
Accuracy:          0.8223
Accuracy (%):      82.23%
----------------------------------------------------------------------------
family level match
Total images:      22524
Identified:        19231
Failed:            3293
Accuracy:          0.85

In [23]:

species_results = (
    df_raw.groupby("file_name")["species_match"]
    .any()
    .reset_index(name="species_identified")
)
df = pd.merge(df_raw, species_results[["file_name", "species_identified"]], on="file_name", how="left")

In [24]:
df_raw.shape

(146955, 78)

In [25]:
species_results.shape

(22524, 2)

In [21]:
df_raw["file_name"].nunique()

22524

# Error Handling

In [27]:
failed_df = df[df["species_identified"] == False]

In [28]:
failed_df.shape

(41690, 79)

In [29]:
failed_df["file_name"].nunique()

6535

## Adding a remarks and needs to be reviewed column

In [45]:
failed_df["remarks"] = None
failed_df["review_needed"] = False

## what is the taxon_rank level for the failed ones?

In [46]:
unique_files_df = failed_df.drop_duplicates(subset=["file_name"])
taxon_counts = unique_files_df["taxon_rank"].value_counts()
taxon_counts

taxon_rank
species       5539
subspecies     549
genus          243
hybrid          69
complex         37
subgenus        33
variety         25
tribe           20
subfamily       19
form             1
Name: count, dtype: int64

In [47]:
import os 
os.makedirs("error_inspection", exist_ok=True)
print("Directory created succesfully")

Directory created succesfully


In [50]:
failed_subspecies_df = failed_df[(failed_df["taxon_rank"]=="subspecies")][["file_name","taxon_rank","genus_name","model_genus_name","genus_match","scientific_name","species_name","model_species_name","species_match","review_needed","remarks"]]
failed_subspecies_df.to_csv("error_inspection/failed_subspecies_df.csv", index=False)

In [51]:
failed_genus_df = failed_df[(failed_df["taxon_rank"]=="genus")][["file_name","taxon_rank","genus_name","model_genus_name","genus_match","scientific_name","species_name","model_species_name","species_match","review_needed","remarks"]]
failed_genus_df.to_csv("error_inspection/failed_genus_df.csv", index=False)

In [ ]:
list_subspecies_files = failed_subspecies_df["file_name"].unique().to_list()
print("Number of unique file_names in subspecies rank:", len(list_subspecies_files))
print(failed_subspecies_df[failed_subspecies_df["file_name"]==list_subspecies_files

In [ ]:
# Review_needed - manually verified
